# Validacion y Resumen Ejecutivo del Consolidado

---

## 1. Resumen ejecutivo

El **consolidado de indicadores** es una tabla unica que unifica los datos de iniciativas de Vinculacion con el Medio (VcM) provenientes de multiples planillas fuente, en distintos formatos y periodos (2022-2025). Cubre **~600 iniciativas** de tres familias de formato distintas y cuatro años de operacion. Su proposito es alimentar directamente los graficos del dashboard de gestion de VcM, evitando que el equipo de visualizacion tenga que lidiar con la heterogeneidad de las fuentes originales. El consolidado incluye 14 columnas-indicador estandarizadas (codigo, facultad, carrera, instrumento, estado, competencias, participantes, etc.) mas columnas de trazabilidad que permiten rastrear cada dato a su archivo de origen.

In [ ]:
import sys
from pathlib import Path
import warnings

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / ".."]
PROJECT_ROOT = next((p.resolve() for p in _candidates if (p / "src" / "__init__.py").exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 80)

# Cargar consolidado ya generado
CONSOLIDADO_PATH = PROJECT_ROOT / "data" / "clean" / "consolidado_indicadores.parquet"
df = pd.read_parquet(CONSOLIDADO_PATH)

print(f"Consolidado cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Años cubiertos: {sorted(df['_anio'].dropna().unique().astype(int))}")
print(f"Instrumentos: {sorted(df['_instrumento'].unique())}")

---
## 2. Panorama del consolidado

Cifras clave que resumen el volumen y distribucion de los datos.

In [ ]:
print(f"Total de iniciativas: {len(df)}")
print(f"Archivos fuente: {df['_archivo_origen'].nunique()}")
print(f"Facultades: {df['facultad'].nunique()}")
print(f"Carreras: {df['carrera'].nunique()}")
print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Por año
ax = axes[0]
por_anio = df.groupby("_anio").size().reset_index(name="n")
ax.bar(por_anio["_anio"].astype(int), por_anio["n"], color="#42a5f5")
ax.set_xlabel("Año")
ax.set_ylabel("N iniciativas")
ax.set_title("Iniciativas por año")
for i, row in por_anio.iterrows():
    ax.text(row["_anio"], row["n"] + 2, str(row["n"]), ha="center", fontsize=9)

# Por instrumento
ax = axes[1]
por_instr = df["_instrumento"].value_counts()
ax.barh(por_instr.index, por_instr.values, color="#66bb6a")
ax.set_xlabel("N iniciativas")
ax.set_title("Iniciativas por instrumento")

# Por facultad (top 6)
ax = axes[2]
por_fac = df["facultad"].value_counts().head(6)
ax.barh(por_fac.index, por_fac.values, color="#ffa726")
ax.set_xlabel("N iniciativas")
ax.set_title("Top 6 facultades")

plt.tight_layout()
plt.show()

In [ ]:
# Tabla resumen por año e instrumento
pivot = df.pivot_table(index="_instrumento", columns="_anio", aggfunc="size", fill_value=0)
pivot.columns = pivot.columns.astype(int)
pivot["Total"] = pivot.sum(axis=1)
print("Distribucion año x instrumento:")
print(pivot.to_string())

---
## 3. Cobertura por columna y año

Muestra que porcentaje de celdas esta poblado por cada columna-indicador,
global y desglosado por año. Los huecos reflejan la evolucion del formulario de VcM
en el tiempo, no errores del pipeline.

In [ ]:
COLS_INDICADOR = [
    "codigo", "facultad", "carrera", "nombre_iniciativa", "estado_sisav",
    "semestre_ejecucion", "dominios_disciplinares", "competencia_sello",
    "actividad", "ciclo_modelo_educativo", "cantidad_act_planificadas",
    "cantidad_act_ejecutadas", "n_participantes", "ods",
]

# Cobertura global
cob_global = df[COLS_INDICADOR].notna().mean() * 100
print("Cobertura global por columna:")
for col, pct in cob_global.items():
    print(f"  {col:<30s} {pct:5.1f}%")
print()

In [ ]:
# Heatmap de cobertura por año
cob_anio = df.groupby("_anio")[COLS_INDICADOR].apply(lambda g: g.notna().mean() * 100)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cob_anio.T, annot=True, fmt=".0f", cmap="YlGn", ax=ax,
            linewidths=0.5, vmin=0, vmax=100)
ax.set_title("% de cobertura por columna-indicador y año", fontsize=11)
ax.set_xlabel("Año")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Interpretacion de los huecos

Los huecos de cobertura no son errores del pipeline. Reflejan como evoluciono
el formulario de SISAV2 a lo largo del tiempo:

- **ciclo_modelo_educativo**: solo existe en formatos 2024-2025 (Centralizadas, VEDP 2024,
  exports expandidos). En 2022-2023 el campo no se registraba o tenia nomenclatura distinta.
- **cantidad_act_ejecutadas**: solo existe en la familia 2022-2023. Los formatos 2024+ no
  incluyen esta columna porque las convocatorias aun no estaban cerradas al momento del export.
- **n_participantes**: cubre ~87% porque algunas planillas 2024 no tienen el dato desagregado
  en la misma columna.

**Mensaje clave para presentacion**: los huecos son honestidad de datos, no fallas tecnicas.
El pipeline reporta lo que existe; no inventa datos donde la fuente no los tiene.

---
## 4. Chequeo de integridad conceptual: dominios disciplinares

La directora de VcM indico que los dominios disciplinares en VEDP se empezaron a consultar
recien en 2023 (genericos), se especificaron en 2024, y en 2025 se sumo Extension.
Antes de 2023 ese dato no deberia existir.

Sin embargo, el consolidado muestra `dominios_disciplinares` poblado al ~98% incluyendo 2022.
Esto sugiere que el mapeo de la Familia A (planillas 2022-2023) podria estar mapeando
la columna "Area" a `dominios_disciplinares`, mezclando dos conceptos.

In [ ]:
# Valores unicos de dominios_disciplinares por periodo
for anio in sorted(df["_anio"].dropna().unique().astype(int)):
    vals = df[df["_anio"] == anio]["dominios_disciplinares"].dropna().unique()
    print(f"\n=== Año {anio} ({len(vals)} valores unicos) ===")
    for v in sorted(vals)[:15]:
        print(f"  {v[:80]}")
    if len(vals) > 15:
        print(f"  ... y {len(vals) - 15} mas")

In [ ]:
# Comparacion directa: 2022 vs 2024-2025
vals_2022 = set(df[df["_anio"] == 2022]["dominios_disciplinares"].dropna().unique())
vals_2024_25 = set(df[df["_anio"].isin([2024, 2025])]["dominios_disciplinares"].dropna().unique())

solo_en_2022 = vals_2022 - vals_2024_25
solo_en_2024_25 = vals_2024_25 - vals_2022
en_ambos = vals_2022 & vals_2024_25

print(f"Valores SOLO en 2022: {len(solo_en_2022)}")
for v in sorted(solo_en_2022)[:10]:
    print(f"  {v[:80]}")

print(f"\nValores SOLO en 2024-2025: {len(solo_en_2024_25)}")
for v in sorted(solo_en_2024_25)[:10]:
    print(f"  {v[:80]}")

print(f"\nValores en AMBOS periodos: {len(en_ambos)}")
for v in sorted(en_ambos)[:10]:
    print(f"  {v[:80]}")

### Veredicto

Si los valores de 2022 son categorias genericas de area (ej. "Relacion con el Entorno",
"Titulados", "Extension") mientras que 2024-2025 tiene dominios disciplinares especificos
(ej. "Ingenieria Civil", "Administracion"), entonces el mapeo esta **mezclando conceptos**.

**Recomendacion**:
- Poner `dominios_disciplinares` en NaN para 2022-2023 (Familia A), donde el dato real
  no existia y se esta mapeando "Area" erroneamente.
- Registrar esta decision como regla de limpieza (R-010 o similar) en
  `docs/reglas_transformacion.md`.
- La correccion se aplica al pipeline de consolidacion (notebook 04), no aqui.

**Nota**: este diagnostico debe validarse con la contraparte antes de ejecutar la correccion.
Si VcM confirma que la columna "Area" de 2022 es efectivamente un concepto distinto,
se anula el mapeo. Si resulta que era un precursor de dominios disciplinares, se conserva
con una anotacion.

---
## 5. Estado del KPI I19: actividades ejecutadas / planificadas

**Formula oficial**: (actividades ejecutadas / actividades planificadas) x 100, por carrera.

**Disponibilidad**: solo calculable para 2022-2023, porque las convocatorias 2024+ aun
no estaban cerradas al momento del export y no registran actividades ejecutadas.
Esto no es una falla del pipeline; es el estado real de los datos de origen.

In [ ]:
df_i19 = df[
    df["cantidad_act_ejecutadas"].notna() &
    df["cantidad_act_planificadas"].notna() &
    (df["cantidad_act_planificadas"] > 0)
].copy()

df_i19["pct_cumplimiento"] = (
    df_i19["cantidad_act_ejecutadas"] / df_i19["cantidad_act_planificadas"] * 100
).round(1)

print(f"Iniciativas con dato de cumplimiento: {len(df_i19)}")
print(f"Años disponibles: {sorted(df_i19['_anio'].unique().astype(int))}")
print(f"Instrumentos: {sorted(df_i19['_instrumento'].unique())}")
print()
print(f"Cumplimiento promedio: {df_i19['pct_cumplimiento'].mean():.1f}%")
print(f"Cumplimiento mediano: {df_i19['pct_cumplimiento'].median():.1f}%")
print()

# Resumen por año
resumen = df_i19.groupby("_anio").agg(
    n_iniciativas=("codigo", "count"),
    promedio=("pct_cumplimiento", "mean"),
    mediana=("pct_cumplimiento", "median"),
).round(1)
print("Cumplimiento por año:")
print(resumen.to_string())

In [ ]:
# Distribucion del cumplimiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(df_i19["pct_cumplimiento"].clip(0, 200), bins=20, color="#42a5f5", edgecolor="white")
ax.axvline(100, color="red", linestyle="--", alpha=0.7, label="100% = todo ejecutado")
ax.set_xlabel("% cumplimiento")
ax.set_ylabel("N iniciativas")
ax.set_title("Distribucion del cumplimiento I19 (2022-2023)")
ax.legend()

ax = axes[1]
por_instr_i19 = df_i19.groupby("_instrumento")["pct_cumplimiento"].mean().sort_values()
ax.barh(por_instr_i19.index, por_instr_i19.values, color="#66bb6a")
ax.axvline(100, color="red", linestyle="--", alpha=0.7)
ax.set_xlabel("% cumplimiento promedio")
ax.set_title("Cumplimiento promedio por instrumento")

plt.tight_layout()
plt.show()

---
## 6. Limitaciones de origen (no atribuibles al pipeline)

Las siguientes limitaciones fueron confirmadas con la contraparte y corresponden
al estado de los datos en la fuente, no a problemas del procesamiento:

1. **Desglose territorial RM**: el dato de territorio/comuna no esta depurado en
   origen. La informacion existe parcialmente en texto libre pero no como campo
   estructurado confiable.

2. **Instrumento VEDP L1/L2**: la distincion entre Linea 1 y Linea 2 recien se
   inicio en 2025 y aun no esta cerrada. No hay datos historicos con este desglose.

3. **Cobertura creciente por año**: los formularios de SISAV2 fueron evolucionando.
   Los años mas antiguos (2022) tienen menos campos que los recientes (2025).
   Esto es inherente a la fuente.

4. **Actividades ejecutadas (I19)**: solo registradas en planillas 2022-2023.
   Las convocatorias 2024+ aun no cierran o cambiaron el formato de seguimiento.

5. **Participantes desagregados por genero**: solo disponibles en exports
   expandidos 2025. El consolidado actual usa el total agregado.

---
## 7. Estado y proximos pasos

### Listo (entregable actual)

- Consolidado de indicadores exportado a CSV y Parquet (`data/clean/consolidado_indicadores.*`)
- 593 iniciativas, 2022-2025, 14 columnas-indicador estandarizadas
- Pipeline de limpieza funcional para exports planos (src/main.py)
- Documentacion completa: reglas, alcance, diccionario, comparacion de formatos
- Notebooks de exploracion y validacion reproducibles

### En validacion (requiere confirmacion de VcM)

- Mapeo de "Area" (2022-2023) a dominios_disciplinares: probable mezcla de conceptos.
  Pendiente de confirmar con la contraparte si se anula o se conserva con anotacion.
- Tratamiento de NaN en I19 para años sin dato de ejecutadas: reportar como "no disponible"
  vs excluir del calculo.

### Proximos pasos (quincena de julio)

1. **Migrar la consolidacion de notebook a src/**: convertir la logica de
   `04_consolidado_indicadores.ipynb` en un modulo (`src/consolidar.py`) integrado
   al pipeline reproducible, con tests.
2. **Resolver decisiones pendientes con VcM**:
   - Tratamiento de dominios_disciplinares pre-2023
   - Jerarquia de instrumentos (VEDP L1/L2)
   - Formato territorial (si se incorpora)
3. **Entregar al equipo de visualizacion**: el consolidado actual es consumible
   directamente. Los graficos finales los construye el otro equipo sobre este archivo.
4. **Incorporar exports expandidos 2025 al pipeline principal**: extender src/ingest.py
   para el esquema de 48-49 columnas cuando VcM confirme que es el formato definitivo.